# Capstone: your own mini-study
**Quantum Machine Learning · Week 4 (QML for research) · Day 5**

Lesson page: https://sara-aminpour.com/tutorials/qml/w4d5.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

**Capstone: a mini-study, start to finish.** One question, a hypothesis written down before running anything, a
protocol, results with their spread, and an honest verdict. The worked example asks a question close to real
research: when one class is rare, does a quantum kernel find it better than a classical one?

In [ ]:
QUESTION = "When one class is rare, does a quantum-kernel SVM find it better than an RBF-kernel SVM?"
HYPOTHESIS = "No difference beyond the spread; weighting the rare class matters more than the choice of kernel."
PROTOCOL = """digits 3 versus 8, with 8s cut down to about a fifth of the data;
PCA to 4 features and scaling fitted on the training part only;
5 stratified random splits, 70% training and 30% test;
every model tuned by 5-fold cross-validation on balanced accuracy, test set used once;
metrics: balanced accuracy and recall of the rare class (ML week 3), plain accuracy for comparison."""
print("Question:  ", QUESTION)
print("Hypothesis:", HYPOTHESIS)

## The data and the models

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, recall_score, accuracy_score
from sklearn.svm import SVC

digits = load_digits()
rng = np.random.default_rng(0)
threes = np.flatnonzero(digits.target == 3)
eights = rng.choice(np.flatnonzero(digits.target == 8), size=45, replace=False)   # the rare class
idx = np.r_[threes, eights]
X, y = digits.data[idx], (digits.target[idx] == 8).astype(int)
print(f"{len(y)} images: {np.sum(y == 0)} threes and {np.sum(y == 1)} eights ({100 * y.mean():.0f}% rare)")

def zz_map(x, reps=2):
    n = len(x)
    qc = QuantumCircuit(n)
    for _ in range(reps):
        qc.h(range(n))
        for q in range(n):
            qc.p(2 * x[q], q)
        for q in range(n - 1):
            qc.cx(q, q + 1)
            qc.p(2 * (np.pi - x[q]) * (np.pi - x[q + 1]), q + 1)
            qc.cx(q, q + 1)
    return qc

def quantum_kernels(U_tr, U_te, s):
    S_tr = np.array([Statevector(zz_map(s * u)).data for u in U_tr])
    S_te = np.array([Statevector(zz_map(s * u)).data for u in U_te])
    return np.abs(S_tr.conj() @ S_tr.T) ** 2, np.abs(S_te.conj() @ S_tr.T) ** 2

def rbf_kernels(U_tr, U_te, gamma):
    sq = lambda A, B: ((A[:, None, :] - B[None, :, :]) ** 2).sum(-1)
    return np.exp(-gamma * sq(U_tr, U_tr)), np.exp(-gamma * sq(U_te, U_tr))

## Run the study

In [ ]:
def tune_and_test(kernels, knobs, y_tr, y_te, weight):
    """Pick the knob and C with the best cross-validated balanced accuracy, then test once."""
    best = (-1, None, None)
    folds = list(StratifiedKFold(5, shuffle=True, random_state=0).split(np.zeros(len(y_tr)), y_tr))
    for knob in knobs:
        K_tr, K_te = kernels(knob)
        for C in (0.1, 1, 10, 100):
            cv = np.mean([balanced_accuracy_score(y_tr[va], SVC(kernel="precomputed", C=C, class_weight=weight)
                          .fit(K_tr[np.ix_(tr, tr)], y_tr[tr]).predict(K_tr[np.ix_(va, tr)])) for tr, va in folds])
            if cv > best[0]:
                best = (cv, C, (K_tr, K_te))
    _, C, (K_tr, K_te) = best
    pred = SVC(kernel="precomputed", C=C, class_weight=weight).fit(K_tr, y_tr).predict(K_te)
    return balanced_accuracy_score(y_te, pred), recall_score(y_te, pred), accuracy_score(y_te, pred)

results = {}
splits = StratifiedShuffleSplit(n_splits=5, test_size=0.3, random_state=0).split(X, y)
for split, (tr, te) in enumerate(splits):
    pca = PCA(n_components=4).fit(X[tr])
    scaler = MinMaxScaler((0, 1)).fit(pca.transform(X[tr]))
    U_tr, U_te = scaler.transform(pca.transform(X[tr])), scaler.transform(pca.transform(X[te]))
    for weight in (None, "balanced"):
        for name, kernels, knobs in (("quantum kernel", lambda s: quantum_kernels(U_tr, U_te, s), (0.25, 0.5, 1.0)),
                                     ("RBF kernel", lambda g: rbf_kernels(U_tr, U_te, g), (0.3, 1, 3, 10))):
            key = f"{name}, {'weighted' if weight else 'unweighted'}"
            results.setdefault(key, []).append(tune_and_test(kernels, knobs, y[tr], y[te], weight))

print(f"{'model':30s}{'balanced acc.':>17s}{'rare recall':>17s}{'accuracy':>17s}")
for key, rows in results.items():
    r = np.array(rows)
    print(f"{key:30s}" + "".join(f"{m:>11.3f} ± {s:.3f}" for m, s in zip(r.mean(0), r.std(0))))

## Your turn: write the study card
A study isn't finished until it's written down. Fill in the card for the worked example: question, hypothesis,
protocol, result (with spread), verdict, limitations, and one next step. Then copy the template for your own study.

In [ ]:
card = {
    "question": QUESTION,
    "hypothesis": HYPOTHESIS,
    "protocol": PROTOCOL,
    "result": "TODO: the key numbers, with their spread",
    "verdict": "TODO: was the hypothesis supported?",
    "limitations": "TODO: what this study can't tell you",
    "next step": "TODO: the experiment you would run next",
}
for k, v in card.items():
    print(f"{k.upper()}: {v}\n")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
r = {k: np.array(v) for k, v in results.items()}
card = {
    "question": QUESTION,
    "hypothesis": HYPOTHESIS,
    "protocol": PROTOCOL,
    "result": (f"balanced accuracy, weighted: quantum {r['quantum kernel, weighted'][:, 0].mean():.3f} ± {r['quantum kernel, weighted'][:, 0].std():.3f}, "
               f"RBF {r['RBF kernel, weighted'][:, 0].mean():.3f} ± {r['RBF kernel, weighted'][:, 0].std():.3f}; "
               f"unweighted: quantum {r['quantum kernel, unweighted'][:, 0].mean():.3f}, RBF {r['RBF kernel, unweighted'][:, 0].mean():.3f}."),
    "verdict": ("Partly supported. With class weighting, the two kernels are within each other's spread. Without it, the "
                "RBF kernel is ahead by about 5 points of balanced accuracy, more than the spread, and weighting helps the "
                "quantum kernel more (+5 points) than the RBF (+2). How the rare class is treated mattered about as much "
                "as the choice of kernel."),
    "limitations": ("One dataset, 4 qubits on an exact simulator, about 14 rare test images per split, so each rare "
                    "image moves the recall by about 7 points; no shots or noise; one quantum feature map."),
    "next step": "Repeat with shots and with a second rare-class dataset, and add a tuned small network as a third model.",
}
for k, v in card.items():
    print(f"{k.upper()}: {v}\n")